In [97]:
words= open("data/shakespeare_100k.txt", "r").read().splitlines()

In [98]:
words[:10]

['THE',
 'SONNETS',
 'ALL',
 'IS',
 'WELL',
 'THAT',
 'ENDS',
 'WELL',
 'THE',
 'TRAGEDY']

In [99]:
len(words[15])

3

In [100]:

store = 0

for w in range(len(words)):
    if len(words[w]) > len(words[store]):
        store = w

print(store)
print(words[store])

36396
Portotartarossa


# Bigram model :

 In bigram model we are going to predict the next character based on the just the previous char
 
 In order to learn which charaters are likely to follow which chars statistically the simplest way is to use count of there co-occurrence

In [101]:

#this whole code creates char & prevChar pairing and count of their co-occurrence
b={}
for w in words:
    chs=["<S>"] + list(w) + ["<E>"] # doing this stores a very usefull imformation about the characters which begin a word and end a word
    for ch1, ch2 in zip(chs,chs[1:]):
        bigram=(ch1,ch2)
        b[bigram]=b.get(bigram,0)+1

In [102]:
sorted(b.items(),key=lambda kv: -kv[1]) # sorted bigram count in descending order

[(('e', '<E>'), 18270),
 (('<S>', 't'), 11637),
 (('t', 'h'), 10797),
 (('t', '<E>'), 10745),
 (('s', '<E>'), 10557),
 (('h', 'e'), 8605),
 (('d', '<E>'), 8582),
 (('r', '<E>'), 7216),
 (('<S>', 's'), 7048),
 (('<S>', 'a'), 6631),
 (('y', '<E>'), 6480),
 (('n', '<E>'), 6188),
 (('o', 'u'), 6044),
 (('e', 'r'), 5841),
 (('<S>', 'h'), 5638),
 (('<S>', 'm'), 5458),
 (('i', 'n'), 5419),
 (('<S>', 'w'), 5262),
 (('a', 'n'), 4903),
 (('r', 'e'), 4779),
 (('h', 'a'), 4535),
 (('o', '<E>'), 4515),
 (('n', 'd'), 4411),
 (('a', 't'), 3892),
 (('<S>', 'b'), 3876),
 (('<S>', 'i'), 3794),
 (('o', 'r'), 3732),
 (('<S>', 'o'), 3729),
 (('e', 's'), 3540),
 (('i', 's'), 3537),
 (('e', 'n'), 3493),
 (('a', 'r'), 3391),
 (('h', 'i'), 3390),
 (('<S>', 'I'), 3344),
 (('l', '<E>'), 3331),
 (('h', '<E>'), 3315),
 (('v', 'e'), 3269),
 (('<S>', 'f'), 3217),
 (('s', 't'), 3168),
 (('l', 'l'), 3116),
 (('o', 'n'), 3022),
 (('m', 'e'), 2935),
 (('<S>', 'd'), 2919),
 (('e', 'a'), 2896),
 (('i', 't'), 2867),
 (('<S

In [103]:
# pytorch and tensors
import torch

a=torch.zeros((3,5),dtype=torch.int32)
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [104]:
a[1,3]=1
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [105]:
# storing this dictionary as co-occurrence matrix using tensors in pytorch

N = torch.zeros((52,52),dtype=torch.int32) 
# we have 53 unique chars in our dataset including capital small letters and special characters that we introduced

In [106]:
lookup=sorted(list(set(''.join(words)))) #lookup table for string to integer mapping of the chars of our dtaset
stoi={s:i+1 for i,s in enumerate(lookup)}
stoi['.']=0 #instead of two chars <S> <E> we will have '.'
itos={i:s for s,i in stoi.items()}

In [107]:
# initialise N matrix

for w in words:
    chs=["."] + list(w) + ["."] 
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        N[ix1,ix2]+=1

In [108]:
#our co-occurrence matrix
N

tensor([[   0, 2478,  883,  ...,    0, 2450,    2],
        [ 849,    0,   26,  ...,    0,   59,    0],
        [   0,  120,    0,  ...,    0,   93,    0],
        ...,
        [  48,    0,    0,  ...,    0,    0,    0],
        [6480,    0,    0,  ...,    0,    0,    0],
        [  12,    0,    0,  ...,    0,    1,    4]], dtype=torch.int32)

In [136]:
#visualisation of N using matplotlib

# import matplotlib.pyplot as plt
# %matplotlib inline

# plt.figure(figsize=(26,26))
# plt.imshow(N, cmap='Blues')

# for i in range(52):
#     for j in range(52):
#         chstr = itos[i] + itos[j]
#         plt.text(j, i, chstr, ha="center", va="top", color="gray")
#         plt.text(j, i, N[i,j].item(), ha='center', va='bottom', color="gray")

# plt.axis('off');


In [137]:
N[0]

tensor([    0,  2478,   883,  1418,   618,   888,   773,   368,   866,  3344,
          133,   160,   702,   770,   458,  1023,   604,    19,   528,  1410,
         2432,    99,    98,  1436,    20,   373,  6631,  3876,  2495,  2919,
         1381,  3217,  1641,  5638,  3794,   155,   669,  2843,  5458,  2474,
         3729,  2199,   186,  1149,  7048, 11637,   792,   454,  5262,     0,
         2450,     2], dtype=torch.int32)

In [138]:
p= N[0].float()
p= p/p.sum() # making the real numbers as probalities 


In [139]:
g= torch.Generator().manual_seed(214748367) # this is a seed which will generate same random number everytime

ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item() # ix stores the index from p choosen randomly using multinomial distribution over p
itos[ix]

's'

In [ ]:
P=N.float()

In [140]:
# loop to make predictions (although you might not even get one correct word but its fun)
# And unfortunately with a basic bigram we can do mostly this much
for i in range(20):
    out=[]
    ix=0
    while True:
        p=N[ix].float()
        p=p/p.sum()
        ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()

        out.append(itos[ix])

        if ix==0:
            break
    print(''.join(out))

ar.
ONTRD.
thooysingoroouige.
powitharokelofatho.
du.
ves.
k.
whe.
t.
they.
Noo.
foulde.
wimmenoot.
tre.
Rof.
fofofourt.
t.
bribris.
theath.
t.


# Conclusion

Bigram model uses co-occurrence matrix N chooses the first element from row 0(N[0]) using multinomial distribution over row0 then goes to row index of the choosen char and selects any char randomly from the probability distribution of that row and repeats this process to create words untill hits . char

so i bigram we have a 2D matrix for Ngram models we have nD matrix